# Keep a live inventory current: update, delete, and compact

Search indexes are often treated as write-once: to change a row you rebuild. Infino tables are **mutable** — you `append`, `update`, and `delete` rows, and the full-text index and SQL views stay correct, durably, with no rebuild.

This notebook runs a product inventory through its life cycle on [Infino](https://pypi.org/project/infino/): stock it, mark items down, discontinue others, compact the storage, and reopen it to confirm the changes persisted — all over one table, with no second system to keep in sync.

## Setup

In [1]:
# %pip install infino pyarrow datasets

In [2]:
import sys
from pathlib import Path

# Make the repo's shared helpers importable.
sys.path.insert(0, str(Path.cwd().parent))

import shutil

import infino
import pyarrow as pa

from _shared.loaders import load_amazon
from _shared.sql import sql_lit, query

## 1. Stock the inventory

Each product has a title, a searchable description, and metadata: price, rating, category, and store. Updates and deletes need durable storage, so this example writes to a local directory (not the in-memory backend).

In [3]:
products = load_amazon(n=1200)
print(f"loaded {len(products)} products")
p = products[0]
print(f"  {p['title'][:60]!r}  ${p['price']}  {p['category']!r}")

loaded 1200 products
  'GiGi Professional Multi-Purpose Wax Warmer, with See-Through'  $25.87  'All Beauty'


## 2. Index it, in three appends

We add the inventory in three batches — each `append` is one atomic commit. `text` is full-text indexed; the rest are SQL-filterable columns.

In [4]:
DB_DIR = "./inventory_data"
shutil.rmtree(DB_DIR, ignore_errors=True)

db = infino.connect(DB_DIR)
schema = pa.schema([
    pa.field("title", pa.large_utf8(), nullable=False),
    pa.field("text", pa.large_utf8(), nullable=False),
    pa.field("price", pa.float64(), nullable=False),
    pa.field("rating", pa.float64(), nullable=False),
    pa.field("category", pa.large_utf8(), nullable=False),
    pa.field("store", pa.large_utf8(), nullable=False),
])
spec = infino.IndexSpec().fts("text")
table = db.create_table("products", schema, spec)

cols = ["title", "text", "price", "rating", "category", "store"]
types = [pa.large_utf8(), pa.large_utf8(), pa.float64(), pa.float64(), pa.large_utf8(), pa.large_utf8()]
third = len(products) // 3
for start in range(0, len(products), third):
    chunk = products[start:start + third]
    table.append(pa.record_batch(
        [pa.array([p[c] for p in chunk], type=t) for c, t in zip(cols, types)],
        schema=schema,
    ))

n = db.query_sql("SELECT COUNT(*) AS n FROM products").to_pydict()["n"][0]
print(f"indexed {n} products across 3 appends")

indexed 1200 products across 3 appends


## 3. Baseline

A quick SQL snapshot before we change anything — total count and the cheapest price on the shelf.

In [5]:
base = query(db, "SELECT COUNT(*) n, MIN(price) cheapest, MAX(price) priciest FROM products")
print(f"products: {base['n'][0]}   price range: ${base['cheapest'][0]:.2f} - ${base['priciest'][0]:.2f}")

products: 1200   price range: $1.01 - $477.06


## 4. Update: mark an item down

`update(predicate, new_rows)` replaces the matching rows with new ones. We pick a product with a unique title, halve its price, and pass the full replacement row. `MutationStats.matched` reports how many rows changed.

In [6]:
# A title that appears exactly once, so the predicate matches a single row.
uniq = query(db, "SELECT title FROM products GROUP BY title HAVING COUNT(*) = 1 ORDER BY title LIMIT 1")
target = uniq["title"][0]

row = query(db, f"SELECT {', '.join(cols)} FROM products WHERE title = {sql_lit(target)}")
replacement = {c: row[c][0] for c in cols}
old_price = replacement["price"]
replacement["price"] = round(old_price * 0.5, 2)

stats = table.update(f"title = {sql_lit(target)}", [replacement])
print(f"updated {stats.matched} row(s): {target[:50]!r}")
print(f"  price ${old_price:.2f} -> ${replacement['price']:.2f}")

check = query(db, f"SELECT price FROM products WHERE title = {sql_lit(target)}")
print(f"  SQL now reads ${check['price'][0]:.2f}")

updated 1 row(s): '#MG COLAB Dry Shampoo Paradise Fragrance 200ml-Giv'
  price $15.99 -> $8.00


  SQL now reads $8.00


## 5. Delete: a clearance sweep

`delete(predicate)` removes every matching row. We clear out the cheapest tenth of the inventory; the rows leave SQL and full-text search at once, and `MutationStats` reports how many were removed.

In [7]:
prices = sorted(p["price"] for p in products)
cutoff = round(prices[len(prices) // 10], 2)

# Grab one product that is about to be cleared, to confirm it disappears.
doomed = query(db, f"SELECT title FROM products WHERE price < {cutoff} ORDER BY price LIMIT 1")["title"][0]

before = query(db, "SELECT COUNT(*) n FROM products")["n"][0]
stats = table.delete(f"price < {cutoff}")
after = query(db, "SELECT COUNT(*) n, MIN(price) cheapest FROM products")

print(f"clearance: delete price < ${cutoff:.2f}")
print(f"  matched {stats.matched}, removed {stats.n_tombstoned}")
print(f"  count {before} -> {after['n'][0]}   cheapest now ${after['cheapest'][0]:.2f}")

gone = table.token_match("text", doomed.split()[0], projection=["title"]).to_pydict()
still_there = any(t == doomed for t in gone.get("title", []))
print(f"  cleared item still searchable? {still_there}")

clearance: delete price < $6.99
  matched 114, removed 114
  count 1200 -> 1086   cheapest now $6.99
  cleared item still searchable? False


## 6. Compact the storage

After many appends and deletes, storage gets fragmented. `optimize()` merges it into fewer, fuller files — a housekeeping step that never changes query results.

In [8]:
table.optimize()
post = query(db, "SELECT COUNT(*) n FROM products")
print(f"after optimize: {post['n'][0]} products (unchanged), storage compacted")

after optimize: 1086 products (unchanged), storage compacted


## 7. Durability: reopen the inventory

The inventory lives on disk (object storage in production). Drop every handle, reconnect from scratch, and the update and delete are still there — every change was committed, not held in memory.

In [9]:
del table
del db

db = infino.connect(DB_DIR)          # fresh connection, same directory
reopened = db.open_table("products")
persisted = query(db, f"SELECT COUNT(*) n FROM products")["n"][0]
price = query(db, f"SELECT price FROM products WHERE title = {sql_lit(target)}")["price"][0]
print(f"reopened inventory: {persisted} products")
print(f"  the marked-down item persisted at ${price:.2f}")

reopened inventory: 1086 products
  the marked-down item persisted at $8.00


## What just happened

One Infino table went through a full mutable life cycle, durably, with no index rebuild:

- **`append`** — stocked the inventory in three commits.
- **`update`** — replaced a row in place; SQL and search reflected the new price immediately.
- **`delete`** — removed a slice of the inventory; rows left SQL and full-text search at once.
- **`optimize`** — compacted the storage without changing results.
- **reopen** — every change persisted; the table is committed state on disk, not memory.

The same table stays full-text searchable and SQL-queryable throughout — one engine, no separate store to update and keep in sync.